Created by Justin Cooke 02/11/2026

The purpose of this script is to find eta ref at a point where variance is different between HYCOM and the PIES data in HYCOM, and calculate it in two ways. The first is integrate to 2000m depth, the second is integrate to full depth. 

We will plot time-series of the two to compare. We can also take two year increments and plot bottom pressure to compare against CPIES.

In [ ]:
# Import modules

# Sci computing
import numpy as np
import scipy as sp
import seawater as sw
import scipy.sparse.linalg as sla

# Parallel comupting
from dask.distributed import Client, LocalCluster
from dask.diagnostics import ProgressBar

# For Data
import netCDF4 as nc
import xarray as xr

# Plotting stuff
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.gridspec as grdspc
import cmocean as cm

# Gen stuff
from datetime import date
today = date.today()

In [ ]:
# Load using XArray the lat lon and depth data first

ds_latlon = xr.open_dataset("./hycom_data/hycom_latlon.nc")
ds_depth = xr.open_dataset("./hycom_data/hycom_depth.nc")

# Now get the lat, lon, and depth for 22N to 28N and -90W to -83W and depth down to 2000m
ds_lon = ds_latlon['Longitude'][:]
ds_lat = ds_latlon['Latitude'][:]
ds_z = ds_depth['Depth'][:] 

# finding the index in lon array that corresponds to 90W
ind90 = list(np.where(ds_lon >= -90)) 
ind83 = list(np.where(ds_lon >= -83))
nlon90 = ind90[0][0]
nlon83 = ind83[0][0] + 1
lon = ds_lon[nlon90:nlon83]

# finding the indices in lat array that correspond to 22W and 28W to only grab data from this region
ind22 = list(np.where(ds_lat >= 22))
nlat22 = ind22[0][0] 
ind28 = list(np.where(ds_lat >= 28))
nlat28 = ind28[0][0] + 1
lat = ds_lat[nlat22:nlat28]

ind2k = list(np.where(ds_z >= 2000))
n2k = ind2k[0][0] + 1
depth = ds_z[:n2k]


# Want eta ref at a single point
ind8625 = list(np.where(lon >= -86.25))
nlon8625 = ind8625[0][0] 
lon_pt = lon[nlon8625]

ind26 = list(np.where(lat >=26))
nlat26 = ind26[0][0]
lat_pt = lat[nlat26]

print('lon idx =',nlon8625,'and lat idx =',nlat26)

In [ ]:
ds_theta = xr.open_dataset("./hycom_data/hycom_temp_pt.nc",chunks={'MT': 540, 'Depth': 13})
ds_sal = xr.open_dataset("./hycom_data/hycom_sal_pt.nc",chunks={'MT': 540, 'Depth': 13})
ds_ssh = xr.open_dataset("./hycom_data/hycom_ssh_filtered.nc", chunks={'MT': 540, 'Latitude': 83, 'Longitude': 88})

In [ ]:
ssh_pt_ts = ds_ssh['ssh'].isel(Latitude=nlat26,Longitude=nlon8625)

ssh_pt = ssh_pt_ts.compute()

ssh_pt_vals = ssh_pt.values

Nt = ssh_pt_vals.shape

In [ ]:
# Now we need to convert depth to pressure

# First we need to define a function dpth which is converts pressure to depth in m

def dpth(pres_dbar,lat_deg):
    x = np.sin(np.radians(lat_deg))**2
    g = 9.780318 * (1.0 + (5.2788e-3 + 2.36e-5 * x) * x)

    depth_in_m = (((-1.82e-15 * pres_dbar + 2.279e-10) * pres_dbar - 2.2512e-5) * pres_dbar + 9.72659) * pres_dbar / g

    return depth_in_m

def prs(depth_meters, latitude_deg, tol=0.001):
    # Iteratively compute pressure [dbar] from depth [m] and latitude [deg] 

    # Parameters
    # depth_meters : float or np.ndarray
        # Depth in meters
    # latitude_deg : float or np.ndarray
        # Latitude degrees north (-90 to 90)
    # tol          : float, optional

    # Returns
    # pressure : np.ndarray 
        # Pressure in decibar (dbar)
    # iterations : int
        # Number of iterations used to converge

    # Convert inputs to arrays

    depth_meters = np.atleast_1d(depth_meters).astype(float)
    latitude_deg = np.atleast_1d(latitude_deg).astype(float)

    # Broadcast latitude to depth shape if needed
    if latitude_deg.size == 1:
        latitude_deg = np.full_like(depth_meters,latitude_deg)
    elif latitude_deg.shape != depth_meters.shape:
        if latitude_deg.shape[0] == depth_meters.shape[1]:
            latitude_deg = np.tile(latitude_deg, (depth_meters.shape[0],1))
        else: 
            raise ValueError("Latitude and Depth must have compatible dimensions")
        
    # Initialization
    pressure = 1.01 * depth_meters
    converged = False
    max_iters = 20
    iters = 1

    while not converged and iters <= max_iters:
        d = dpth(pressure,latitude_deg)
        new_pressure = pressure + (depth_meters - d) * 1.01
        delta = np.abs(new_pressure - pressure)

        if np.max(delta) < tol:
            converged = True

        pressure = new_pressure
        iters += 1

    if not converged:
        pressure[:] = np.nan

    return pressure.squeeze()

In [ ]:
# Need an mxn array where m = length(depth) and n = length(lat)
this_depth = np.meshgrid(depth.values,lat_pt)
this_fulldepth = np.meshgrid(ds_z.values,lat_pt)

# Transpose this
this_depth_tr = np.transpose(this_depth[0])
this_fulldepth_tr = np.transpose(this_fulldepth[0])

# Get our depth in dbar now
(m2dbar) = prs(this_depth_tr,lat_pt)
(m2dbar_fd) = prs(this_fulldepth_tr,lat_pt)

del this_depth, this_depth_tr, this_fulldepth, this_fulldepth_tr

# Create a temporary pressure array which copies m2dbar at all latitudes and depths along longitude
# Resulting array size is lon x depth x lat
temp_P = np.tile(m2dbar,(1,1,1))
temp_P_fd = np.tile(m2dbar_fd,(1,1,1))
# Reshape the array so it is depth x lat x lon
temp_P = np.moveaxis(temp_P,[0,1,2],[-1,-3,-2]) 
temp_P_fd = np.moveaxis(temp_P_fd,[0,1,2],[-1,-3,-2])

# Now we will extend the array along the temporal axis for each member 
# This has dimensions time [days] x depth x lat x lon
p = np.tile(temp_P,(Nt[0],1,1,1))
p_fd = np.tile(temp_P_fd,(Nt[0],1,1,1))

del temp_P, temp_P_fd

# Finally need reference pressure array with all zeros
p_ref = np.zeros_like(p)
p_ref = np.squeeze(p_ref)
p_fd_ref = np.zeros_like(p_fd) 
p_fd_ref = np.squeeze(p_fd_ref)

# squeeze pressure
p = np.squeeze(p)
p_fd = np.squeeze(p_fd)


In [ ]:
print(p.shape,p_ref.shape,p_fd.shape,p_fd_ref.shape)

In [ ]:
# Get values at depth
sal_pt = ds_sal['salinity'].values[:,:-7,0,0]
sal_pt_fd = ds_sal['salinity'].values[:,:-6,0,0]
theta_pt = ds_theta['temperature'].values[:,:-7,0,0]
theta_pt_fd = ds_theta['temperature'].values[:,:-6,0,0]

temp_pt = sw.eos80.temp(sal_pt,theta_pt,p,p_ref)
temp_pt_fd = sw.eos80.temp(sal_pt_fd,theta_pt_fd,p_fd[:,:-6],p_fd_ref[:,:-6])

In [ ]:
s35 = np.full_like(sal_pt,35.0)  # e.g., reference salinity
s35_fd = np.full_like(sal_pt_fd,35.0)
T0  = np.zeros_like(theta_pt)  # e.g., reference temp
T0_fd  = np.zeros_like(theta_pt_fd)

In [ ]:
sv_350p = 1/sw.eos80.dens(s35,T0,p)
sv_350p_fd = 1/sw.eos80.dens(s35_fd,T0_fd,p_fd[:,:-6])

sv = 1/sw.eos80.dens(sal_pt,temp_pt,p)
sv_fd = 1/sw.eos80.dens(sal_pt_fd,temp_pt_fd,p_fd[:,:-6])

svan = sv - sv_350p
svan_fd = sv_fd - sv_350p_fd

In [ ]:
m2dbar_fd

In [ ]:
# integrate

# Convert pressure in decibars to Pascals (N/m^2) 
p_Pa = m2dbar * 1e4
p_Pa_fd = m2dbar_fd[:-6] * 1e4

# The difference in gravity at each latitude is small so I'm just going to average it
# The change in magnitude for gravity as a function of latitude is much much smaller than that for pressure, which is why we cared
# for it previously 
g_real = np.mean(sw.eos80.g(lat_pt))

# For the integration, we are performing: int_0^2024e4 Pa (1/dens(s,t,p) - 1/dens(35psu,0C,p)) dp

gpan = np.trapezoid(svan,p_Pa)
gpan_fd = np.trapezoid(svan_fd,p_Pa_fd)

In [ ]:
phi_g = gpan/g_real
phi_g_fd = gpan_fd/g_real

eref = ssh_pt.values - phi_g[-1]
eref_fd = ssh_pt.values - phi_g_fd[-1]

eref = eref - np.mean(eref)
eref_fd = eref_fd - np.mean(eref_fd)

print(eref,eref_fd)

In [ ]:
plt.plot(eref)
plt.plot(eref_fd)